# 05a — Clustering KMeans

3 versiones × 9 K (2-10) = 27 modelos. MiniBatchKMeans para N>50k.

In [1]:
import pandas as pd
import numpy as np
import time, joblib
from pathlib import Path
from datetime import datetime
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder

ROOT = Path.cwd().parents[1] if Path.cwd().name == 'fase4_gustos' else Path.cwd()
DATA_QC = ROOT / 'data' / 'data_qc_gustos'
DATA_MODELS = ROOT / 'data' / 'data_models_gustos'
INFORMES_BASE = ROOT / 'informes' / 'fase4_gustos' / '04_clustering'
DATA_MODELS.mkdir(parents=True, exist_ok=True)

MASTER_VERSIONS = {
    'v1_conservative': DATA_QC / 'master_table_gustos_v1_conservative.parquet',
    'v2_intermediate': DATA_QC / 'master_table_gustos_v2_intermediate.parquet',
    'v3_aggressive':   DATA_QC / 'master_table_gustos_v3_aggressive.parquet',
}

def reduce_high_card(df, max_unique=10):
    out = df.copy()
    for cat in out.select_dtypes(include=['object', 'category']).columns.tolist():
        n = out[cat].nunique(dropna=True)
        if n > max_unique:
            top = out[cat].value_counts().head(max_unique).index.tolist()
            out[cat] = out[cat].where(out[cat].isin(top), 'OTHER')
    return out

def preprocess_master(master_df, nan_threshold_zero=0.30):
    """Returns (X_scaled, preprocessor, feature_names_out)."""
    df = master_df.drop(columns=['user_id']).copy()
    numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
    categorical_cols = df.select_dtypes(include=['object','category']).columns.tolist()
    nan_pcts = df[numeric_cols].isna().mean()
    num_low = nan_pcts[nan_pcts < nan_threshold_zero].index.tolist()
    num_high = nan_pcts[nan_pcts >= nan_threshold_zero].index.tolist()
    print(f'  num_low_nan: {len(num_low)} | num_high_nan: {len(num_high)} | cat: {len(categorical_cols)}')
    df = reduce_high_card(df, max_unique=10)
    preproc = ColumnTransformer([
        ('num_low', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', RobustScaler())]), num_low),
        ('num_high', Pipeline([('imp', SimpleImputer(strategy='constant', fill_value=0)), ('sc', RobustScaler())]), num_high),
        ('cat', Pipeline([('imp', SimpleImputer(strategy='constant', fill_value='missing')), ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), categorical_cols),
    ])
    X = preproc.fit_transform(df)
    feature_names = preproc.get_feature_names_out()
    return X, preproc, feature_names

def load_and_preprocess(version_name):
    master = pd.read_parquet(MASTER_VERSIONS[version_name])
    print(f'[{version_name}] shape={master.shape}')
    user_ids = master['user_id'].values
    X, preproc, names = preprocess_master(master)
    return X, user_ids, preproc, names, master.shape

from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

REPORT = INFORMES_BASE / '05a_kmeans'
REPORT.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42
K_RANGE = list(range(2, 11))
SAMPLE_FOR_SILHOUETTE = 20000  # silhouette score sobre subsample (caro O(n^2))
results = []


In [2]:

for version in MASTER_VERSIONS:
    print(f'\n=== {version} ===')
    X, user_ids, preproc, names, shape = load_and_preprocess(version)
    print(f'  X shape post-preproc: {X.shape}')
    # Guardar preprocessor (uno por versión)
    joblib.dump(preproc, DATA_MODELS / f'preprocessor_{version}.joblib')

    # Subsample for silhouette
    rng = np.random.RandomState(RANDOM_STATE)
    sil_idx = rng.choice(len(X), min(SAMPLE_FOR_SILHOUETTE, len(X)), replace=False)
    X_sil = X[sil_idx]

    inertias = []; silhouettes = []
    for k in K_RANGE:
        t0 = time.time()
        if len(X) > 50_000:
            km = MiniBatchKMeans(n_clusters=k, random_state=RANDOM_STATE, batch_size=2048, n_init=3)
        else:
            km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
        labels = km.fit_predict(X)
        # silhouette sobre subsample
        labels_sub = km.predict(X_sil) if hasattr(km, 'predict') else labels[sil_idx]
        try:
            sil = float(silhouette_score(X_sil, labels_sub)) if len(set(labels_sub)) > 1 else float('nan')
        except Exception:
            sil = float('nan')
        try:
            db = float(davies_bouldin_score(X, labels))
        except Exception:
            db = float('nan')
        try:
            ch = float(calinski_harabasz_score(X, labels))
        except Exception:
            ch = float('nan')
        inertias.append(km.inertia_)
        silhouettes.append(sil)
        results.append({
            'algorithm': 'KMeans', 'version': version, 'k': k,
            'silhouette': sil, 'davies_bouldin': db, 'calinski_harabasz': ch,
            'inertia': float(km.inertia_),
            'n_clusters_actual': int(len(set(labels))),
            'elapsed_s': time.time()-t0,
        })
        print(f'  K={k}: sil={sil:.4f} db={db:.4f} ch={ch:.0f} t={time.time()-t0:.1f}s')
        # Guardar modelo
        joblib.dump({'model': km, 'labels': labels, 'user_ids': user_ids, 'preproc': preproc}, DATA_MODELS / f'kmeans_{version}_k{k}.joblib')

    # Plot elbow + silhouette
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(K_RANGE, inertias, marker='o'); axes[0].set_xlabel('K'); axes[0].set_ylabel('Inertia'); axes[0].set_title(f'Elbow — {version}'); axes[0].grid(alpha=0.3)
    axes[1].plot(K_RANGE, silhouettes, marker='o', color='C1'); axes[1].set_xlabel('K'); axes[1].set_ylabel('Silhouette'); axes[1].set_title(f'Silhouette — {version}'); axes[1].grid(alpha=0.3); axes[1].axhline(0.15, color='red', linestyle='--', alpha=0.4, label='umbral 0.15'); axes[1].legend()
    plt.tight_layout()
    plt.savefig(REPORT / f'elbow_silhouette_{version}.png', dpi=120, bbox_inches='tight')
    plt.close()
    print(f'plot {version} guardado')



=== v1_conservative ===


[v1_conservative] shape=(114412, 115)
  num_low_nan: 86 | num_high_nan: 24 | cat: 4


/var/folders/0f/3zq36dr1451_ttvt93n947ph0000gw/T/ipykernel_1706/175113424.py:36: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=['object','category']).columns.tolist()
/var/folders/0f/3zq36dr1451_ttvt93n947ph0000gw/T/ipykernel_1706/175113424.py:25: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence thi

  X shape post-preproc: (114412, 132)


  K=2: sil=0.9601 db=1.2190 ch=8335 t=2.5s


...

  K=6: sil=0.0961 db=1.5175 ch=1661 t=1.8s


  K=7: sil=0.4177 db=1.3050 ch=1350 t=2.3s


  K=8: sil=0.1617 db=1.7644 ch=1241 t=2.2s


  K=9: sil=0.3204 db=1.1899 ch=5349 t=2.2s


  K=10: sil=0.4546 db=1.1136 ch=2089 t=2.3s
plot v3_aggressive guardado


In [3]:

res_df = pd.DataFrame(results)
res_df.to_csv(REPORT / 'metrics.csv', index=False)
print(f'\nmetrics.csv ({len(res_df)} filas)')
print(res_df[['version','k','silhouette','davies_bouldin','calinski_harabasz']].to_string(index=False))

best_per_version = res_df.sort_values('silhouette', ascending=False).groupby('version').head(1)
lines = [f'# 05a KMeans', f'**Fecha**: {datetime.now().strftime("%Y-%m-%d %H:%M")}', '',
         '## K óptimo por versión (max silhouette)', '',
         '| Versión | K | silhouette | Davies-Bouldin | Calinski-Harabasz |',
         '|---|---:|---:|---:|---:|']
for _, r in best_per_version.iterrows():
    lines.append(f"| {r['version']} | {int(r['k'])} | {r['silhouette']:.4f} | {r['davies_bouldin']:.4f} | {r['calinski_harabasz']:.0f} |")
(REPORT / 'execution_report.md').write_text('\n'.join(lines))
print('execution_report.md')



metrics.csv (27 filas)
        version  k  silhouette  davies_bouldin  calinski_harabasz
v1_conservative  2    0.960110        1.219035        8335.131662
v1_conservative  3    0.807673        1.233677        4259.992279
v1_conservative  4    0.613709        1.233296        2757.807871
v1_conservative  5    0.561921        1.518787        2048.606739
v1_conservative  6    0.133927        1.483337        1661.470480
v1_conservative  7    0.412303        1.302855        1349.891215
v1_conservative  8    0.123854        1.558692        1241.372039
v1_conservative  9    0.187029        1.259561        5348.942752
v1_conservative 10    0.444307        1.154986        2088.971308
v2_intermediate  2    0.960111        1.219035        8335.131666
v2_intermediate  3    0.807703        1.233677        4259.992280
v2_intermediate  4    0.614402        1.233249        2758.844571
v2_intermediate  5    0.567693        1.514113        2048.606206
v2_intermediate  6    0.128496        1.489849      